# TDMEC Embedding Pilot (Kaggle GPU)\n\n**Model name:** TDMEC (Temporal Dynamic Multiplex Evolutionary Community)\n\nThis notebook runs the bounded Qwen3 embedding pilot (default 10k+10k), Stage-B pooling,\ngraph–text alignment, and `TDMEC_INPUT` export.\n\nLabels: `PROVISIONAL_SMOKE_ONLY` / `ENGINEERING_VALIDATION` / `NOT_FOR_FINAL_THESIS_CONCLUSIONS`\n\nRestartable: re-run from the top after setting dataset paths. No Google Drive dependency.

In [ ]:
# 1) Install dependencies (Kaggle GPU session)\nimport os, sys, subprocess\nfrom pathlib import Path\n\nCODE_ROOT = Path('/kaggle/working/community-evolution-modeling')\n# Adjust if your private code dataset mounts elsewhere\nCANDIDATES = [\n    Path('/kaggle/input/tdmec-embedding-code'),\n    Path('/kaggle/input/tdmec-embedding-code-20260804'),\n    Path('/kaggle/working'),\n]\nprint('Looking for code package…')\nfor c in CANDIDATES:\n    print(' ', c, 'exists' if c.exists() else 'missing')

In [ ]:
# Unpack transfer tarball if present, else assume repo already extracted\nimport tarfile, glob\n\nWORKING = Path('/kaggle/working')\nREPO = WORKING / 'community-evolution-modeling'\ntarballs = list(Path('/kaggle/input').glob('**/tdmec_embedding_code_transfer_*.tar.gz'))\nif tarballs and not (REPO / 'src' / 'tdmec_embeddings').exists():\n    print('Extracting', tarballs[0])\n    with tarfile.open(tarballs[0], 'r:gz') as tf:\n        tf.extractall(WORKING)\nprint('REPO', REPO, 'exists=', REPO.exists())\nsys.path.insert(0, str(REPO / 'src'))\nos.chdir(REPO)\n\nreq = REPO / 'requirements' / 'embeddings-target-studio.txt'\nif req.exists():\n    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])\nelse:\n    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'safetensors', 'psutil', 'pyyaml', 'pyarrow', 'numpy'])

In [ ]:
# 2) Resolve Kaggle dataset mounts (private)\ndef find_run(root_name_parts, run_id):\n    for p in Path('/kaggle/input').rglob('manifest.json'):\n        try:\n            import json\n            m = json.loads(p.read_text())\n            if m.get('run_id') == run_id:\n                return p.parent\n        except Exception:\n            pass\n    raise FileNotFoundError(run_id)\n\nDATASET_A = find_run(['smoke', 'a'], 'smoke_a_pg_001')\nDATASET_B = find_run(['smoke', 'b'], 'smoke_b_pg_001')\nOUT = Path('/kaggle/working/tdmec_embeddings')\nOUT.mkdir(parents=True, exist_ok=True)\n\nos.environ['TDMEC_DATASET_A_ROOT'] = str(DATASET_A)\nos.environ['TDMEC_DATASET_B_ROOT'] = str(DATASET_B)\nos.environ['TDMEC_EMBEDDING_OUTPUT_ROOT'] = str(OUT)\n# Immutable pin from successful preflight\nREV = '5cf2132abc99cad020ac570b19d031efec650f2b'\nos.environ['QWEN3_MODEL_REVISION'] = REV\nos.environ['QWEN3_TOKENIZER_REVISION'] = REV\nprint('A', DATASET_A)\nprint('B', DATASET_B)\nprint('OUT', OUT)

In [ ]:
# 3) Optional: dry-run source verification\nfrom tdmec_embeddings.run_pilot_embedding import main as pilot_main\nrc = pilot_main([\n    '--config', 'configs/qwen3_tdmec_pilot.yaml',\n    '--dry-run',\n])\nassert rc == 0, rc

In [ ]:
# 4–7) Pilot embed + pool + align + export TDMEC_INPUT\n# Dual authorization is mandatory. Do not remove these flags casually.\nrc = pilot_main([\n    '--config', 'configs/qwen3_tdmec_pilot.yaml',\n    '--authorize-real-model',\n    '--authorize-bounded-pilot',\n    # '--resume',  # uncomment after interruption\n])\nassert rc == 0, rc

In [ ]:
# 8) Final package validation report\nimport json\nfrom pathlib import Path\npkgs = sorted(Path('/kaggle/working/tdmec_embeddings').glob('TDMEC_INPUT_*'))\nprint('packages', pkgs)\nassert pkgs, 'TDMEC_INPUT package not found'\npkg = pkgs[-1]\nfrom tdmec_embeddings.validation import validate_tdmec_input_package\nreport = validate_tdmec_input_package(pkg, expected_dimension=512)\nprint(json.dumps(report, indent=2)[:4000])\nassert report['passed'], report.get('failures')\nprint('READY:', pkg)